# Which lines of evidence predict recognised biology? (notebook 113)

**The question.** Notebook 112 showed that the number of independent lines calling a gene predicts whether the
gene is in a named literature, disease, transporter or hormone set. That validates the count. It does not say
which lines carry the information, and it weighs every line the same.

**The frozen protocol** is `docs/results/pt-evidence-model.md` (commit b629793).
- Q1: do the lines add to expression in predicting recognition, held out one annotation source at a time?
- Q2: does the fitted weighting beat notebook 112's plain count?
- Q3: which lines carry the information?
- Q4: the relabeled control.
- Q5: the prioritised unanchored list.

**Leakage rule.** A line built from the same source as the held-out annotation is dropped for that fold.

**Sections.** 1 setup; 2 features and outcomes; 3 Q1 and Q2 folds; 4 Q3 coefficients; 5 Q4 relabeled control;
6 Q5 the ranked list; 7 verdicts; 8 figures; 9 reading and run record.

## 1 · Setup and inputs

In [ ]:
import argparse
import json
import os
import sys
import time
from pathlib import Path

NOTEBOOK_START = time.perf_counter()
starts = [Path.cwd()] + ([Path(__file__).resolve().parent] if '__file__' in globals() else [])
project = next(p for s in starts for p in (s, *s.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
T112 = results_root / 'pt_gene_atlas' / 'tables'
T91 = results_root / 'pt_positional_decomposition' / 'tables'
T107 = results_root / 'pt_injury_heterogeneity' / 'tables'
T111 = results_root / 'pt_androgen_s3' / 'tables'
output = results_root / 'pt_evidence_model'
tables, figures = output / 'tables', output / 'figures'
for folder in (tables, figures):
    folder.mkdir(parents=True, exist_ok=True)

NOTEBOOK_LOGIC_VERSION = '113.evidence_model.1'
SEED = 113
BOOTSTRAP, C_PENALTY, AUC_GAIN = 1000, 1.0, .05
L91_CUT, M91_CUT = .5, .4
for path in (T112 / 'aki_atlas.csv.gz', T112 / 'species_atlas.csv.gz', T91 / 'i_genes.csv.gz',
             T107 / 'genes.csv.gz', T111 / 'sets.csv'):
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path}')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from IPython.display import display

from pseudospace import gene_atlas as ga

MM = 1 / 25.4
INK, MUTED, LIGHT = '#0b0b0b', '#52514e', '#d9d8d4'
BLUE, ORANGE, GREEN, GREY = '#0072B2', '#D55E00', '#009E73', '#999999'
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5,
    'axes.titleweight': 'bold', 'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6,
    'legend.fontsize': 6, 'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': .6, 'xtick.major.width': .6, 'ytick.major.width': .6, 'xtick.major.size': 2.5,
    'ytick.major.size': 2.5, 'lines.linewidth': 1.0, 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_rows', 80)


def save(fig, stem):
    for suffix in ('png', 'pdf'):
        fig.savefig(figures / f'{stem}.{suffix}', bbox_inches='tight')


guard_log = []
ATLAS = {name: pd.read_csv(T112 / f'{key}.csv.gz').set_index('gene')
         for name, key in (('AKI', 'aki_atlas'), ('species', 'species_atlas'))}
for name, a in ATLAS.items():
    print(name, a.shape, 'lines', [c for c in a.columns if c.startswith('call ')])
guard_log.append('G1: ' + ', '.join(f'{n} {len(a):,} genes' for n, a in ATLAS.items()))

## 2 · Features and outcomes

Features: each line as 0/1, each line's testable mask, and nine expression-decile indicators. Outcomes: each
annotation source in turn. The leakage rule drops a line fitted from the source being held out.

In [ ]:
LEAK = {'is androgen set': 'call A9 androgen', 'is handoff': 'call S7 handoff'}


def features_of(atlas):
    calls = atlas[[c for c in atlas.columns if c.startswith('call ')]].astype(float)
    testable = atlas[[c for c in atlas.columns if c.startswith('testable ')]].astype(float)
    dec = pd.get_dummies(atlas['expression decile'].astype(int), prefix='decile').astype(float)
    dec = dec.drop(columns=dec.columns[0])                      # the lowest decile is the reference
    X = pd.concat([calls, testable, dec], axis=1)
    return X, list(calls.columns), list(dec.columns)


FEATURES, LINE_COLS, DEC_COLS, SOURCES = {}, {}, {}, {}
for name, a in ATLAS.items():
    X, lines, dec = features_of(a)
    FEATURES[name], LINE_COLS[name], DEC_COLS[name] = X, lines, dec
    SOURCES[name] = [c for c in a.columns if c.startswith('is ')]
    print(name, 'features', X.shape[1], '· sources', SOURCES[name])
    print('   source sizes', {s: int(a[s].sum()) for s in SOURCES[name]})

## 3 · Q1 and Q2 · Held-out prediction of recognition

In [ ]:
rows = []
for name, a in ATLAS.items():
    X, lines, dec = FEATURES[name], LINE_COLS[name], DEC_COLS[name]
    usable = a['testable lines'] > 0
    for held in SOURCES[name]:
        train_sources = [s for s in SOURCES[name] if s != held]
        y_train = a[train_sources].any(axis=1)
        # test genes: not positive in any training source, so a gene cannot be positive on both sides
        test_mask = usable & ~y_train
        y_test = a.loc[test_mask, held]
        if y_test.sum() < 10 or (~y_test).sum() < 10:
            rows.append({'atlas': name, 'source': held, 'positives': int(y_test.sum()), 'note': 'too few'})
            continue
        drop = [LEAK[held]] if held in LEAK and LEAK[held] in X.columns else []
        cols_full = [c for c in X.columns if c not in drop]
        cols_expr = list(dec)
        count = (X[[c for c in lines if c not in drop]].sum(axis=1)).rename('lines')
        cols_count = None
        tr = usable & ~a[held]                                   # train on genes outside the held-out source
        fits = {
            'full': (X.loc[tr, cols_full], X.loc[test_mask, cols_full]),
            'expression': (X.loc[tr, cols_expr], X.loc[test_mask, cols_expr]),
            'count': (pd.concat([count, X[cols_expr]], axis=1).loc[tr],
                      pd.concat([count, X[cols_expr]], axis=1).loc[test_mask]),
        }
        out = {'atlas': name, 'source': held, 'positives': int(y_test.sum()), 'tested': int(test_mask.sum())}
        for tag, (Xtr, Xte) in fits.items():
            coef, score = ga.evidence_model(Xtr, y_train.loc[tr], C=C_PENALTY, seed=SEED)
            out[f'auc {tag}'] = roc_auc_score(y_test, score(Xte))
        out['full - expression'] = out['auc full'] - out['auc expression']
        out['full - count'] = out['auc full'] - out['auc count']
        rows.append(out)
FOLDS = pd.DataFrame(rows)
FOLDS.to_csv(tables / 'fold_auc.csv', index=False)
display(FOLDS.round(3))

aki_folds = FOLDS[FOLDS.atlas.eq('AKI') & FOLDS['auc full'].notna()]
Q1_HOLDS = bool((aki_folds['full - expression'] >= AUC_GAIN).sum() >= 3 and
                all(FOLDS[FOLDS.atlas.eq(n)]['full - expression'].mean() > 0 for n in ATLAS))
Q2_HOLDS = bool(FOLDS['full - count'].mean() > 0)
print('Q1 holds:', Q1_HOLDS, '· Q2 holds:', Q2_HOLDS)

## 4 · Q3 · Which lines carry the information?

In [ ]:
# **Leakage correction (post hoc).** The frozen leakage rule drops a line only in the fold where its own source is
# held out, so an all-sources fit would still score the androgen line against the androgen sets and the handoff
# line against the handoff annotation. Both fits are reported; the bootstrap is run on the corrected one, which is
# the one to read.

LEAK_SOURCE = {'AKI': 'is androgen set', 'species': 'is handoff'}
rng = np.random.default_rng(SEED)
COEF = {}
for name, a in ATLAS.items():
    X, lines = FEATURES[name], LINE_COLS[name]
    usable = a['testable lines'] > 0
    Xu = X.loc[usable]
    y_all = a.loc[usable, SOURCES[name]].any(axis=1)
    keep_sources = [s for s in SOURCES[name] if s != LEAK_SOURCE[name]]
    y = a.loc[usable, keep_sources].any(axis=1)
    coef_all, _ = ga.evidence_model(Xu, y_all, C=C_PENALTY, seed=SEED)
    coef, score = ga.evidence_model(Xu, y, C=C_PENALTY, seed=SEED)
    boots = np.zeros((BOOTSTRAP, len(lines)))
    idx = np.arange(len(Xu))
    for b in range(BOOTSTRAP):
        take = rng.choice(idx, len(idx), replace=True)
        cb, _ = ga.evidence_model(Xu.iloc[take], y.iloc[take], C=C_PENALTY, seed=SEED)
        boots[b] = cb[lines].to_numpy()
    tab = pd.DataFrame({'line': [l.replace('call ', '') for l in lines],
                        'calls': [int(a[l].sum()) for l in lines],
                        'coefficient': coef[lines].to_numpy(),
                        'lo': np.quantile(boots, .025, axis=0), 'hi': np.quantile(boots, .975, axis=0),
                        'coefficient (all sources, leaking)': coef_all[lines].to_numpy()})
    tab['atlas'] = name
    COEF[name] = (tab, coef, score)
    print(f'\n{name} (outcome without {LEAK_SOURCE[name]}):')
    display(tab.sort_values('coefficient', ascending=False).round(3))
COEF_TABLE = pd.concat([v[0] for v in COEF.values()], ignore_index=True)
COEF_TABLE.to_csv(tables / 'coefficients.csv', index=False)
CLEAN_COEF = COEF_TABLE.rename(columns={'coefficient': 'coefficient (leaking source removed)',
                                        'coefficient (all sources, leaking)': 'coefficient (all sources)'})
CLEAN_COEF.to_csv(tables / 'coefficients_leak_corrected.csv', index=False)

## 5 · Q4 · The relabeled control

The level, location, variability and androgen lines are rebuilt with a balanced relabeling in the condition's
place, using notebook 112's § C1 rule, and the AKI model is refitted.

In [ ]:
g91 = pd.read_csv(T91 / 'i_genes.csv.gz').set_index('gene')
g91 = g91[~g91.index.duplicated()]
het = pd.read_csv(T107 / 'genes.csv.gz').set_index('gene')
ar = pd.read_csv(T111 / 'sets.csv', index_col=0)
ar_maintained = ar[['maintained S1', 'maintained S2', 'maintained S3']].any(axis=1)
AKI = ATLAS['AKI'].index


def beyond_others(stat, others, cutoff, ratio=.5):
    s = pd.Series(stat).astype(float).abs()
    ok = s >= cutoff
    for o in others:
        ok &= pd.Series(o).astype(float).abs() <= ratio * s
    return ok.fillna(False)


REL = []
for tag, other in (('R1', 'R2'), ('R2', 'R1')):
    a = ATLAS['AKI'].copy()
    repl = {
        'call A1 level': g91.index[beyond_others(g91[f'L {tag}'], [g91['L'], g91[f'L {other}']], L91_CUT)],
        'call A2 location': g91.index[beyond_others(g91[f'M {tag}'], [g91['M'], g91[f'M {other}']], M91_CUT)],
        'call A8 variability': het.index[beyond_others(het[tag], [het['Δε condition'], het[other]], 0.)],
        'call A9 androgen': ar.index[ar_maintained & (ar[f'L {tag}'] < 0)],
    }
    for col, genes in repl.items():
        a[col] = AKI.isin(genes)
    X, lines, dec = features_of(a)
    usable = a['testable lines'] > 0
    y = a.loc[usable, SOURCES['AKI']].any(axis=1)
    coef, score = ga.evidence_model(X.loc[usable], y, C=C_PENALTY, seed=SEED)
    y_cor = a.loc[usable, [s for s in SOURCES['AKI'] if s != LEAK_SOURCE['AKI']]].any(axis=1)
    coef_cor, _ = ga.evidence_model(X.loc[usable], y_cor, C=C_PENALTY, seed=SEED)
    REL.append(pd.DataFrame({'line': [l.replace('call ', '') for l in lines], 'relabeling': tag,
                             'calls': [int(a[l].sum()) for l in lines],
                             'coefficient': coef[lines].to_numpy(),
                             'coefficient (leak corrected)': coef_cor[lines].to_numpy()}))
    # held-out AUC on the same folds, full model only
    for held in SOURCES['AKI']:
        train_sources = [s for s in SOURCES['AKI'] if s != held]
        y_train = a[train_sources].any(axis=1)
        test_mask = usable & ~y_train
        y_test = a.loc[test_mask, held]
        if y_test.sum() < 10:
            continue
        drop = [LEAK[held]] if held in LEAK and LEAK[held] in X.columns else []
        cols = [c for c in X.columns if c not in drop]
        tr = usable & ~a[held]
        cf, sc = ga.evidence_model(X.loc[tr, cols], y_train.loc[tr], C=C_PENALTY, seed=SEED)
        REL.append(pd.DataFrame([{'line': f'AUC {held}', 'relabeling': tag, 'calls': int(y_test.sum()),
                                  'coefficient': roc_auc_score(y_test, sc(X.loc[test_mask, cols]))}]))
RELABELED = pd.concat(REL, ignore_index=True)
RELABELED.to_csv(tables / 'coefficients_relabeled.csv', index=False)
lines_only = RELABELED[~RELABELED.line.str.startswith('AUC')]
pivot = lines_only.pivot(index='line', columns='relabeling', values='coefficient (leak corrected)')
pivot['condition'] = CLEAN_COEF[CLEAN_COEF.atlas.eq('AKI')].set_index('line')['coefficient (leaking source removed)']
display(pivot.round(3))
Q4_LINES = ['A1 level', 'A2 location', 'A8 variability', 'A9 androgen']
Q4_HOLDS = bool(all(abs(pivot.loc[l, t]) < abs(pivot.loc[l, 'condition'])
                    for l in Q4_LINES for t in ('R1', 'R2')))
print('Q4 holds (relabeled coefficients smaller on every rebuilt line, leak-corrected outcome):', Q4_HOLDS)

## 6 · Q5 · The prioritised unanchored list

In [ ]:
RANKED = {}
for name, a in ATLAS.items():
    tab, coef, score = COEF[name]
    usable = a['testable lines'] > 0
    anchored = a[SOURCES[name]].any(axis=1)
    sub = a[usable & ~anchored & a['clean']].copy()
    sub['evidence score'] = score(FEATURES[name].loc[sub.index])
    sub = sub.sort_values('evidence score', ascending=False)
    keep = [c for c in ('evidence score', 'lines', 'supported', 'L', 'M', 'Kirita 12 h t', 'KPMP t',
                        'injury class', 'injury location', 'survival', 'A', 'E mouse', 'E human',
                        'zonation class') if c in sub.columns]
    calls = [c for c in sub.columns if c.startswith('call ')]
    RANKED[name] = sub[keep + calls]
    RANKED[name].to_csv(tables / f'ranked_unanchored_{name.lower()}.csv', index_label='gene')
    print(f'\n{name}: {len(sub):,} clean unanchored genes ranked; top 30:')
    display(sub.head(30)[keep].round(2))

# recognition rate by score decile, against the count model
CALIB = []
for name, a in ATLAS.items():
    tab, coef, score = COEF[name]
    usable = a['testable lines'] > 0
    s = pd.Series(score(FEATURES[name].loc[usable]), index=a.index[usable])
    anchored = a.loc[usable, SOURCES[name]].any(axis=1)
    for tag, v in (('model', s), ('count', a.loc[usable, 'lines'])):
        d = pd.qcut(v.rank(method='first'), 10, labels=False)
        for q, idx in anchored.groupby(d).groups.items():
            CALIB.append({'atlas': name, 'model': tag, 'decile': int(q),
                          'recognition rate': float(anchored.loc[idx].mean()), 'genes': len(idx)})
CALIB = pd.DataFrame(CALIB)
CALIB.to_csv(tables / 'calibration.csv', index=False)
display(CALIB.pivot_table(index='decile', columns=['atlas', 'model'], values='recognition rate').round(3))

In [ ]:
auc_rows = RELABELED[RELABELED.line.str.startswith('AUC')]
REL_AUC = float(auc_rows['coefficient'].mean())
COND_AUC = float(aki_folds['auc full'].mean())
print(f'relabeled held-out AUC {REL_AUC:.3f} against the condition model {COND_AUC:.3f}')
TESTABLE_SOURCES = int(FOLDS[FOLDS.atlas.eq('AKI')]['auc full'].notna().sum())
Q1_AMONG_TESTABLE = bool((aki_folds['full - expression'] >= AUC_GAIN).sum() >= 2 and TESTABLE_SOURCES == 3)
VERDICTS = pd.DataFrame([
    ('Q1 lines add to expression (>= 0.05 AUC in 3 of 5 AKI sources, positive mean on both atlases)', Q1_HOLDS),
    (f'Q1 as far as it can be tested: only {TESTABLE_SOURCES} AKI sources leave held-out positives, '
     f'{int((aki_folds["full - expression"] >= AUC_GAIN).sum())} of them clear 0.05', Q1_AMONG_TESTABLE),
    ('Q2 the fitted weighting beats the plain count on average', Q2_HOLDS),
    ('Q4 relabeled lines carry smaller coefficients', Q4_HOLDS),
    ('Q4 the relabeled model predicts recognition worse (held-out AUC, mean over sources)',
     bool(REL_AUC < COND_AUC)),
], columns=['verdict', 'holds'])
VERDICTS.to_csv(tables / 'verdicts.csv', index=False)
display(VERDICTS)

## 7 · Figures

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(130 * MM, 48 * MM))
for ax, name in zip(axes, ('AKI', 'species')):
    sub = FOLDS[FOLDS.atlas.eq(name) & FOLDS['auc full'].notna()]
    x = np.arange(len(sub))
    for i, (tag, colour) in enumerate((('auc expression', LIGHT), ('auc count', GREY), ('auc full', BLUE))):
        ax.bar(x + (i - 1) * .26, sub[tag] - .5, bottom=.5, color=colour, width=.25)
    ax.axhline(.5, color=MUTED, lw=.5)
    ax.set_xticks(x, [s.replace('is ', '') for s in sub['source']], rotation=30, ha='right')
    ax.set_ylim(.4, 1.0); ax.set_title(name)
axes[0].set_ylabel('held-out AUC')
save(fig, '113a_fold_auc')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(95 * MM, 55 * MM))
tab = COEF['AKI'][0].sort_values('coefficient').reset_index(drop=True)
y = np.arange(len(tab))
ax.hlines(y, tab['lo'], tab['hi'], color=GREY, lw=1)
ax.plot(tab['coefficient'], y, 'o', ms=3, color=BLUE)
for tag, colour, off in (('R1', GREEN, .18), ('R2', ORANGE, -.18)):
    r = RELABELED[(RELABELED.relabeling == tag) & ~RELABELED.line.str.startswith('AUC')].set_index('line')
    ax.plot(r.reindex(tab['line'])['coefficient (leak corrected)'], y + off, 'd', ms=2.5, color=colour)
ax.axvline(0, color=MUTED, lw=.5)
ax.set_yticks(y, tab['line']); ax.set_xlabel('coefficient')
save(fig, '113b_coefficients')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(120 * MM, 45 * MM))
for ax, name in zip(axes, ('AKI', 'species')):
    sub = CALIB[CALIB.atlas.eq(name)]
    for tag, colour in (('count', GREY), ('model', BLUE)):
        s = sub[sub.model.eq(tag)]
        ax.plot(s['decile'], s['recognition rate'], marker='o', ms=2.5, color=colour)
    ax.set_xlabel('score decile'); ax.set_title(name)
axes[0].set_ylabel('share in a named set')
save(fig, '113c_calibration')
plt.show()

## 8 · How the protocol was read, and the run record

In [ ]:
READING = pd.DataFrame([
    ('reading', 'Features are each line as 0/1, each line\'s testable mask, and nine expression-decile indicators; the lowest decile is the reference.'),
    ('reading', 'Each fold trains on genes outside the held-out source with the other sources as positives, and tests on genes positive in no training source.'),
    ('reading', 'The leakage rule drops the androgen line when the androgen sets are held out and the handoff line when the handoffs are.'),
    ('reading', 'The count model is notebook 112\'s line count plus the same expression deciles, so the comparison is weighting, not information.'),
    ('reading', 'Bootstrap intervals resample genes 1,000 times and refit; the intervals are percentile intervals of the line coefficients.'),
    ('reading', 'The ranked list keeps only genes passing notebook 112\'s clean filters and in no named set.'),
], columns=['kind', 'what'])
READING.to_csv(tables / 'how_the_protocol_was_read.csv', index=False)

RUN = {'notebook': '113_pt_evidence_model', 'logic_version': NOTEBOOK_LOGIC_VERSION, 'seed': SEED,
       'guards': guard_log, 'bootstrap': BOOTSTRAP,
       'q1_holds': Q1_HOLDS, 'q2_holds': Q2_HOLDS, 'q4_holds': Q4_HOLDS,
       'folds': FOLDS.round(4).to_dict('records'),
       'coefficients': COEF_TABLE.round(4).to_dict('records'),
       'ranked': {k: int(len(v)) for k, v in RANKED.items()},
       'seconds': round(time.perf_counter() - NOTEBOOK_START, 1)}
(output / 'run_record.json').write_text(json.dumps(RUN, indent=2, default=str))
print(json.dumps({k: RUN[k] for k in ('q1_holds', 'q2_holds', 'q4_holds', 'ranked', 'seconds')}, indent=2, default=str))